# POC 1 projet

- Creation : *16/12/2024*

Réalisation d'une preuve de reproduction de bout en bout :
1. [x] chargement d'une image initiale
1. [x] chargement d'un modèle (AlexNet, VGG16)
    1. [x] modification du modèle pour le rendre compatible avec la méthode
        1. [x] dérivation de classes existantes pour le chargement des poids
        1. [x] modification des attributs `return_indices` des instances `MaxPool2D`
        1. [x] récupération des switches durant le `forward`
1. [x] récupération d'une carte de caractéristiques dans le bloc "feature" du CNN
    1. [x] visualisation simple
1. [x] nettoyage après détection de la plus forte activation
1. [x] application du DeconvNet correspondant
    1. [x] avec ou sans les flips des noyaux de convolution
    1. [x] déconvolution de la carte de caractéristiques nettoyée
    1. [x] utilisation des switches récupérés
1. [x] affichage du résultat d'une déconvolution complète dans l'espace des pixels
1. [x] calcul de la zone réceptive d'une activation dans l'espace des pixels
1. [x] cadrage de cette zone dans l'image initiale

> *Note 18/02/2025*
> 
> Codes réutilisés, améliorés et maintenant <span style="color:red">obsolètes</span> (voire buggués) dans ce carnet

## Modules

In [ ]:
# Modules prédéfinis
import os
from typing import Any, Dict, List, Optional, Tuple, TypeVar

import torch
import torch.nn as nn
import torchvision
from torchvision import transforms as T
from torchvision.io import decode_image

import torchinfo

import matplotlib.pyplot as plt
import matplotlib.patches as patches

print("PyTorch ver:", torch.__version__)
print("TorchVision ver:", torchvision.__version__)
print("TorchInfo ver:", torchinfo.__version__)

In [ ]:
# Modules locaux
from datasets import __version__ as datasets_version, DATASET_0, DATASET_2, get_title
from utils import __version__ as utils_version, display_image_tensor as display_image_tensor_, display_pictures_grid, UnNormalize, get_receptive_field_in_pixel_space, get_output_sizes

print("datasets ver:", utils_version)
print("utils ver:", datasets_version)

## Environnement

In [ ]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

In [4]:
# Spécifiquement pour un carnet de type Jupyter
def display_image_tensor(img_tensor, verbose=True):
    display_image_tensor_(img_tensor, verbose=verbose, fn_display=display)

## 1. Chargement d'une image initiale

In [ ]:
DataPath = DATASET_2["path"]
ImageFile = "n02128757_snow_leopard.JPEG"

filename = os.path.join(DataPath, ImageFile)
img_tensor = decode_image(filename) # Format : torch.Tensor

print(get_title(filename.split("/")[-1], DataPath))
display_image_tensor(img_tensor)

## 2. Chargement d'un modèle CNN : Alexnet

### 2.1. Dérivation du modèle pour la visualisation des caractéristiques

- Initialisation de l'attribut `return_indices` de chaque Pool2d à `True`.
- Modification de `forward` pour disposer en retour des indices des `switch indices`.


A partir de l'analyse des codes :
- [`fn torchvision.model.alexnet.alexnet` ver 0.19](https://pytorch.org/vision/0.19/_modules/torchvision/models/alexnet.html#alexnet)
- [`class torchvision.models.alexnet.AlexNet` ver 0.19]()

In [6]:
# From https://github.com/pytorch/vision/blob/main/torchvision/models/_utils.py
# Necessary in alexnetfordev (evolvment of models.alexnet)
V = TypeVar("V")
def _ovewrite_named_param(kwargs: Dict[str, Any], param: str, new_value: V) -> None:
    if param in kwargs:
        if kwargs[param] != new_value:
            raise ValueError(f"The parameter '{param}' expected value {new_value} but got {kwargs[param]} instead.")
    else:
        kwargs[param] = new_value

In [7]:
class AlexNetForDeconv(torchvision.models.AlexNet):
    """
    Evolvement of AlexNet class
    """
    def __init__(self, num_classes: int = 1000, dropout: float = 0.5) -> None:
        super().__init__(num_classes, dropout)
        self.set_return_indices(False)
    

    def forward(self, 
                x: torch.Tensor,
                idx_layer: Optional[int] = None,
                verbose: bool = False
                ) -> torch.Tensor | Tuple[torch.Tensor, List[Tuple[int, torch.Tensor]]]:
        """
        If self.return_indices, return the forward result AND the collection of (#i, tensor of switch indices) for each MaxPool2d

        Args:
            x (tensor): input for forward
            idx_stop (int, optional): indice of the module from which to get the ouput, if set

        Returns:
            
        """

        if idx_layer != None:
            if idx_layer < 0:
                idx_layer += len(self.features)
            assert (0 <= idx_layer) and (idx_layer < len(self.features)), f"i should be in [-{len(self.features)}; {len(self.features)}["

            self.set_return_indices(True)
            switch_indices = []

            #Playing a part of x = self.features(x)
            for i, m in enumerate(self.features):
                if verbose:
                    print(f"[{i}] forward ", m)

                if isinstance(m, nn.MaxPool2d):
                    x, indices = m(x)
                    switch_indices.append((i, indices))
                else:
                    x = m(x)
                
                if verbose:
                    print("\t x.size:", x.size())
                          
                if i == idx_layer:
                    break

            self.set_return_indices(False) # Restore default state
            return x, switch_indices
        else:
            self.set_return_indices(False)
            return super().forward(x)


    def set_return_indices(self, return_indices: bool) -> None:
        """
        Change the return_indices attribut of each Pool2d in self.features module group (CNN part)
        """
        self.return_indices = return_indices
        for m in self.features:
            if isinstance(m, nn.MaxPool2d):
                m.return_indices = return_indices


def alexnetfordeconv(*,
                  weights: Optional[torchvision.models.AlexNet_Weights] = None,
                  progress: bool = True,
                  **kwargs: Any
                  ) -> AlexNetForDeconv:
    """
    Evolvement of models.alexnet function
    """
    weights = torchvision.models.AlexNet_Weights.verify(weights)

    if weights is not None:
        _ovewrite_named_param(kwargs, "num_classes", len(weights.meta["categories"]))

    model = AlexNetForDeconv(**kwargs)

    if weights is not None:
        model.load_state_dict(weights.get_state_dict(progress=progress, check_hash=True))

    return model

In [ ]:
model_alexnet = alexnetfordeconv(weights='IMAGENET1K_V1')
model_alexnet.eval()
print(model_alexnet)
assert model_alexnet.return_indices == False, "model_alexnet.return_indices should be False by default"

In [ ]:
torchinfo.summary(model_alexnet, input_size=torch.Size([1, 3, 224, 224]), mode="eval")

## 3. Récupération d'un bloc de caractéristiques

### 3.1 Visualisation simple

Applications des transformations à l'image d'entrée, comme précaunisé pour l'utilisation d'Alexnet (méthode explicite, une méthode implicite peut-être disponible dans PyTorch).

In [10]:
imagenet_mean = DATASET_0["means"]
imagenet_std = DATASET_0["stds"]

geo_transforms = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
])

transforms = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.Lambda(lambda t: t/255.), # because read_image -> [0..255]
    T.Normalize(mean=imagenet_mean, std=imagenet_std),
])

Seules les transformations géométriques sont appliquées.

In [ ]:
img_tensor_tfd_geo = geo_transforms(img_tensor)
display_image_tensor(img_tensor_tfd_geo)

Toutes les transformations géométriques sont appliquées.

In [ ]:
img_tensor_tfd = transforms(img_tensor)
display_image_tensor(img_tensor_tfd)

Test sur la cohérence des dimensions des valeurs retournées par le forward dans le cas de génération des indices de switch.

In [ ]:
batch_input = img_tensor_tfd.unsqueeze(dim=0)
print("Input (size) : ", batch_input.size())

all_expected = [
    (0, [1, 64, 55, 55], 0),
    (2, [1, 64, 27, 27], 1),
    (7, [1, 384, 13, 13], 2),
    (-1, [1, 256, 6, 6], 3)
]
for idx_layer, expected, len_indices in all_expected:
    output, switch_indices = model_alexnet(batch_input, idx_layer=idx_layer)
    print("Output (size) :", output.size(), "\tSwitch indices :", len(switch_indices))
    assert torch.Size(expected) == output.size(), f"Size should be {expected}"
    assert len_indices == len(switch_indices), f"Len(size) shoud be {len_indices}"

Affichage de cartes de caractéristiques à la sortie du deuxième module de la partie `features` du modèle, c'est-à-dire la couche 1 (base 0), la sortie du premier `ReLU`.

In [ ]:
output_l1, switch_indices = model_alexnet(batch_input, idx_layer=1, verbose=True)
output = output_l1.detach()
display_pictures_grid(
    output.squeeze(dim=0).reshape((output.size(1), 1, output.size(2), output.size(3))),
    # .squeeze pour supprimer la partie "batch"
    # .reshape(...,1,...) pour insérer la dimension channel (à 1 car monochrome)
    per_rows=8,
    titles=range(output.size(1))
    )

In [ ]:
# Elements pour le debugage du pb avec max_unpool2d : affichage des info du max dans le layer 1
print(output.size())
idx_max = torch.argmax(output).item()
chn_max = idx_max // (output.size(-2) * output.size(-1))
idx_max_in_chn = idx_max % (output.size(-2) * output.size(-1))
row_max = idx_max_in_chn // output.size(-1)
col_max = idx_max_in_chn % output.size(-1)
print(idx_max, chn_max, idx_max_in_chn, row_max, col_max)
print(output[0, chn_max, row_max, col_max].item())

Affichage de cartes de caractéristiques à la sortie du troisième module de la partie `features` du modèle, c'est-à-dire la couche 2 (base 0), la sortie du premier `MaxPool2d`.

In [ ]:
output_l2, switch_indices_l2 = model_alexnet(batch_input, idx_layer=2, verbose=True)
output = output_l2.detach()
display_pictures_grid(
    output.squeeze(dim=0).reshape((output.size(1), 1, output.size(2), output.size(3))),
    # .squeeze pour supprimer la partie "batch"
    # .reshape(...,1,...) pour insérer la dimension channel (à 1 car monochrome)
    per_rows=8,
    titles=range(output.size(1))
    )

In [ ]:
# Elements pour le debugage du pb avec max_unpool2d : retrouver les infos dans les indices
print(len(switch_indices_l2))
indices = switch_indices_l2[0][1]
print(type(indices.size()))

print(output.size())
idx_max = torch.argmax(output).item()
chn_max = idx_max // (output.size(-2) * output.size(-1))
idx_max_in_chn = idx_max % (output.size(-2) * output.size(-1))
row_max = idx_max_in_chn // output.size(-1)
col_max = idx_max_in_chn % output.size(-1)
print(chn_max, row_max, col_max)
print(output[0, chn_max, row_max, col_max].item())
print(indices[0, chn_max, row_max, col_max])
print((indices[0, chn_max] == 680).nonzero(as_tuple=False))


## 4. Nettoyage des cartes de caractéristiques

Prise en compte du problème technique lié au fonctionnement de l'`Unpool` qui peut écraser le résultat préservation de la caractéristique activée, selon sa position.

In [18]:
##TODO : généralisation en intégrant la dimension batch
##TODO : que se passe-t-il si plusieurs même max ?
def clean_feature_maps(
        feature_maps: torch.Tensor,
        idx_map: Optional[int] = None,
        indices_from_pool2d: Optional[torch.Tensor] = None
        ) -> Tuple[torch.Tensor, float, torch.Tensor]:
    """
    Args:
        - feature_maps (torch.Tensor): assuming size (Channels, Row, Col)
        - idx_map (int, optional) : indice of a specific map 
        - indices_from_pool2d (torch.Tensor, optional) : if provided, feature maps is generated by a pool2d, indices assuming size (Channels, Row, Col)

    Returns:
        - cleaned feature maps, all zeros tensor but the max value(s)
        - max value
        - Tensor([[chn_max, row_max, col_max]]) : coords of the max
    """
    cleaned = torch.zeros_like(feature_maps)
    if idx_map == None:
        idx_max = torch.argmax(feature_maps).item()
        chn_max = idx_max // (feature_maps.size(-2) * feature_maps.size(-1))
        idx_max_in_chn = idx_max % (feature_maps.size(-2) * feature_maps.size(-1))
    else:
        if idx_map < 0:
            idx_map += feature_maps.size(-3)
            assert (0 <= idx_layer) and (idx_layer < feature_maps.size(-3)), f"i should be in [-{feature_maps.size(-3)}; {feature_maps.size(-3)}[" 
        chn_max = idx_map
        idx_max_in_chn = torch.argmax(feature_maps[idx_map]).item()
        idx_max = (idx_map + 1) * (feature_maps.size(-2) * feature_maps.size(-1)) + idx_max_in_chn

    row_max = idx_max_in_chn // feature_maps.size(-1)
    col_max = idx_max_in_chn % feature_maps.size(-1)
    
    if indices_from_pool2d != None:
        # Keep the activation that will not be overloaded by unpool process -> max indices in feature map <=> last of list returned by nonzero
        coords = (indices_from_pool2d[chn_max] == indices_from_pool2d[chn_max, row_max, col_max]).nonzero()
        coord = coords[-1]
        cleaned[chn_max, coord[-1], coord[-2]] = feature_maps[chn_max, row_max, col_max]
        coords = coord.unsqueeze(dim=0)
    else:
        coords = torch.tensor([[chn_max, row_max, col_max]])
        cleaned[chn_max, row_max, col_max] = feature_maps[chn_max, row_max, col_max]


    return cleaned, cleaned[chn_max, row_max, col_max].item(), coords

Nettoyage de la carte de caractéristiques **contenant la plus forte activation de toute** la couche L1.

In [ ]:
cleaned_output, max_value, max_coord = clean_feature_maps(output_l1.detach().squeeze(dim=0))
cleaned_output = cleaned_output.unsqueeze(dim=0)

print(max_value, max_coord)
display_pictures_grid(
    cleaned_output.squeeze(dim=0).reshape(
        (cleaned_output.size(1), 1, cleaned_output.size(2), cleaned_output.size(3))
        ),
    per_rows=8,
    titles=range(output.size(1))
    )

Nettoyage de la carte de caractéristiques **14** de la couche L1.

In [ ]:
cleaned_output, max_value, max_coord = clean_feature_maps(output_l1.detach().squeeze(dim=0), idx_map=14)
cleaned_output = cleaned_output.unsqueeze(dim=0)

print(max_value, max_coord)
display_pictures_grid(
    cleaned_output.squeeze(dim=0).reshape(
        (cleaned_output.size(1), 1, cleaned_output.size(2), cleaned_output.size(3))
        ),
    per_rows=8,
    titles=range(output.size(1))
    )

Nettoyage de la carte de caractéristiques 14 de la couche L2, générée par un MaxPool2d.

In [ ]:
indices = switch_indices_l2[0][1]
cleaned_output, max_value, max_coord = clean_feature_maps(
    output_l2.detach().squeeze(dim=0), indices_from_pool2d=indices.squeeze(dim=0)
    )
cleaned_output = cleaned_output.unsqueeze(dim=0)

print(max_value, max_coord)
display_pictures_grid(
    cleaned_output.squeeze(dim=0).reshape(
        (cleaned_output.size(1), 1, cleaned_output.size(2), cleaned_output.size(3))
        ),
    per_rows=8,
    titles=range(output.size(1))
    )

## 5. Application d'un Deconvnet correspondant

Création d'une fonction qui appliquera les éléments de déconvolution de façon symétrique à l'architecture CNN, et selon l'indication de la couche dont on récupère la carte de caractéristique.

In [25]:
def apply_deconvnet(conv_model: nn.Module,
                    x: torch.Tensor,
                    idx_layer: int,
                    flip_kernels: bool = False,
                    use_bias: bool = False,
                    clean_feature_map: bool = True,
                    idx_map: Optional[int] = None,
                    verbose: bool = False
                    ) -> torch.Tensor|Tuple[torch.Tensor, torch.Tensor]:
    """
    Args:
        - conv_model
        - x (torch.Tensor) : input to forward until idx_layer module (size [B, C, H, W])
        - idx_layer (int)
        - flip_kernels
        - copy_bias
        - 

    Returns:
        - backward deconvolution result
        - coords of max activation, required for receptive field process, if clean_feature_map
    
    """
    if idx_layer < 0:
        idx_layer += len(conv_model.features)
    assert (0 <= idx_layer) and (idx_layer < len(conv_model.features)), f"i should be in [-{len(conv_model.features)}; {len(conv_model.features)}["
    
    # Generate feature map and switch indices
    output, switch_indices = conv_model(x, idx_layer, verbose)
    if verbose:
        print("forwarded output.size:", output.size(), "len(switch_indices):", len(switch_indices))
        print(output.min().item(), output.max().item())

    # Clean feature map
    if clean_feature_map:
        indices = None
        if isinstance(conv_model.features[idx_layer], nn.MaxPool2d):
            _, indices = switch_indices[-1]
            indices = indices.squeeze(dim=0)

        output, max_activation, max_coords = clean_feature_maps(
            output.squeeze(dim=0), idx_map, indices_from_pool2d=indices
            )
        output = output.unsqueeze(dim=0)

    for i, module in zip(range(idx_layer, -1, -1), reversed(conv_model.features[:idx_layer+1])):
        if verbose:
            print(f"[{i}] reverse of ", module)
        
        if isinstance(module, nn.MaxPool2d):
            _, indices = switch_indices.pop()
            output = nn.functional.max_unpool2d(
                output,
                indices=indices,
                kernel_size=module.kernel_size,
                stride=module.stride,
                padding=module.padding
            )
        elif isinstance(module, nn.ReLU):
            output = nn.functional.relu(output)
        elif isinstance(module, nn.Conv2d):
            weight = torch.flip(module.weight, [2, 3]) if flip_kernels else module.weight
            if use_bias and module.bias != None:
                bias = module.bias.unsqueeze(dim=0).unsqueeze(dim=0).reshape(module.bias.size(0), 1, 1).unsqueeze(dim=0)
                output -= bias
            output = nn.functional.conv_transpose2d(
                output,
                weight=weight,
                stride=module.stride,
                padding=module.padding,
                output_padding=1 if module.stride[0] > 1 else 0, # Because stride > 1
                dilation=module.dilation
            )

        if verbose:
            print(f"\t\t> output.size", output.size(), output.min().item(), output.max().item())

    if clean_feature_map:
        return output, max_coords
    else:
        return output

## 6. Affichage du résultat d'une déconvolution complète dans l'espace des pixels

In [26]:
idx_layer = 2

### Sans nettoyage de la carte de caractéristiques

In [ ]:
output_deconv = apply_deconvnet(conv_model=model_alexnet,
                                x=batch_input,
                                idx_layer=idx_layer,
                                flip_kernels=True,
                                use_bias=True,
                                clean_feature_map=False,
                                verbose=False
                                )

print("batch_input.size:", batch_input.size())
print("output_deconv.size", output_deconv.size())

In [ ]:
display_image_tensor(output_deconv.squeeze(dim=0))

Dénormalisation puis rétablissement de la distribution dans $[0, 1]$.

In [ ]:
unnormalizer = UnNormalize(imagenet_mean, imagenet_std)
output_deconv_un = unnormalizer(output_deconv.squeeze(dim=0))
display_image_tensor(output_deconv_un)

b_output_un = output_deconv_un
b_output_un = b_output_un - b_output_un.min()
b_output_un = b_output_un / b_output_un.max()
display_image_tensor(b_output_un)

### Avec nettoyage de la carte de caractéristiques

In [ ]:
output_deconv, max_coords = apply_deconvnet(conv_model=model_alexnet,
                                            x=batch_input,
                                            idx_layer=idx_layer,
                                            flip_kernels=True,
                                            use_bias=True,
                                            clean_feature_map=True,
                                            verbose=False
                                            )

output_deconv = output_deconv.squeeze(dim=0).detach()
display_image_tensor(output_deconv)

print("max_coords", max_coords)
print("Dénormalisation puis rétablissement de la distribution dans [0, 1].")

unnormalizer = UnNormalize(imagenet_mean, imagenet_std)
output_deconv_un = unnormalizer(output_deconv)
display_image_tensor(output_deconv_un)

b_output_un = output_deconv_un
b_output_un = b_output_un - b_output_un.min()
b_output_un = b_output_un / b_output_un.max()
display_image_tensor(b_output_un)

## 7. Calcul de la zone réceptive d'une activation dans l'espace des pixels

In [ ]:
input_size=torch.Size([1, 3, 224, 224])
output_sizes = get_output_sizes(model_alexnet.features, input_size=input_size)
idx_layer = 2

pos = (max_coords[0][0].item(), max_coords[0][1].item())
print(f"Activation {pos} in layer {idx_layer}")
pixel_space_size= (input_size[-2], input_size[-1])
receptive_field = get_receptive_field_in_pixel_space(
    pos=pos,
    idx_layer=idx_layer, # 3rd ReLU output
    cnn_modules=model_alexnet.features,
    output_sizes=output_sizes,
    pixel_space_size=pixel_space_size
)
print(receptive_field)

## 8. Cadrage de cette zone dans l'image initiale

In [ ]:
print(type(img_tensor_tfd_geo), img_tensor_tfd_geo.size())
display_image_tensor(img_tensor_tfd_geo)                        

In [36]:
(top, left), (bottom, right) = receptive_field

In [ ]:
from PIL import ImageDraw
img_receptive_field = T.functional.to_pil_image(img_tensor_tfd_geo)
img1 = ImageDraw.Draw(img_receptive_field)
print(receptive_field)
img1.rectangle((left, top, right, bottom), outline ="green") 

display(img_receptive_field)

In [ ]:
#rf_min, rf_max = receptive_field

img_crop = img_receptive_field.crop((left, top, right, bottom))
display(img_crop)